# Medallion Architecture in Databricks (Azure) – Complete Guide

**Medallion Architecture** is a **data design pattern** used in Databricks and Delta Lake to organize data into multiple quality layers.

The goal is to **improve data quality step by step** while keeping raw data safe and creating reliable datasets for analytics and machine learning.

It consists of three main layers:

* 🥉 Bronze (Raw Data)
* 🥈 Silver (Cleaned & Validated Data)
* 🥇 Gold (Business-Ready Data)

---

# Why Do We Need Medallion Architecture?

Imagine an e-commerce company receives data from multiple sources:

* Website orders
* Mobile app orders
* ERP system
* CRM system
* Third-party APIs

The incoming data may contain:

* Duplicate records
* Missing values
* Invalid dates
* Incorrect formats
* Corrupt files

If this raw data is used directly for reporting, dashboards will be inaccurate.

Instead, we process the data in stages.

---

# Medallion Architecture Overview

```text
                Source Systems
      ┌────────────────────────────────┐
      │ SQL Server                     │
      │ Oracle                         │
      │ SAP                            │
      │ REST APIs                      │
      │ CSV / JSON Files               │
      └────────────────────────────────┘
                  │
                  ▼
         Azure Data Factory
                  │
                  ▼
         Azure Data Lake Storage
                  │
                  ▼
        ┌────────────────────────┐
        │ Bronze Layer           │
        │ Raw Data               │
        └────────────────────────┘
                  │
                  ▼
        ┌────────────────────────┐
        │ Silver Layer           │
        │ Cleaned Data           │
        └────────────────────────┘
                  │
                  ▼
        ┌────────────────────────┐
        │ Gold Layer             │
        │ Business Aggregations  │
        └────────────────────────┘
                  │
                  ▼
      Power BI / ML / SQL Analytics
```

---

# Bronze Layer (Raw Layer)

## Purpose

The Bronze layer stores **raw data exactly as received** from source systems.

No major transformations are performed.

Think of Bronze as the **backup copy** of your source data.

---

## Characteristics

* Raw data
* Original schema
* No business rules
* No joins
* No aggregations
* Append-only in most cases
* Supports replay if downstream processing fails

---

## Example

Source CSV:

| CustomerID | Name | City     |
| ---------- | ---- | -------- |
| 101        | John | New York |
| 102        | Sara | London   |
| 102        | Sara | London   |

The duplicate is stored as-is in Bronze.

---

## Bronze Folder Structure

```text
adls
│
└── bronze
      │
      ├── customers
      ├── orders
      ├── products
      └── payments
```

---

## Real-Time Banking Example

Every night:

```text
Core Banking System

↓

Customer Transactions

↓

ADF

↓

Bronze Layer
```

Nothing is cleaned.

Even invalid transactions are stored for audit purposes.

---

# Silver Layer (Cleaned Layer)

## Purpose

The Silver layer contains **validated, cleaned, and enriched data**.

This is where most data engineering work happens.

---

## Typical Transformations

* Remove duplicates
* Handle NULL values
* Standardize date formats
* Data type conversion
* Join multiple datasets
* Filter invalid records
* Data validation
* Incremental processing
* Merge updates

---

## Example

Bronze:

| CustomerID | Name | City     |
| ---------- | ---- | -------- |
| 101        | John | New York |
| 102        | Sara | London   |
| 102        | Sara | London   |

Silver:

| CustomerID | Name | City     |
| ---------- | ---- | -------- |
| 101        | John | New York |
| 102        | Sara | London   |

Duplicate removed.

---

## Silver Folder

```text
silver
│
├── customers
├── orders
├── products
└── payments
```

---

## Real-Time Retail Example

Data comes from:

* POS system
* Online store
* Mobile app

In Silver:

* Remove duplicate customers
* Standardize product IDs
* Validate payment status
* Convert currencies if needed

---

# Gold Layer (Business Layer)

## Purpose

Gold contains **business-ready data** optimized for reporting, dashboards, and machine learning.

Business calculations are performed here.

---

## Transformations

* Aggregations
* KPIs
* Star schema
* Fact tables
* Dimension tables
* Business metrics

---

## Example

Silver Orders

| OrderID | CustomerID | Amount |
| ------- | ---------- | ------ |
| 1       | 101        | 500    |
| 2       | 101        | 300    |
| 3       | 102        | 200    |

Gold Customer Summary

| CustomerID | TotalSales |
| ---------- | ---------- |
| 101        | 800        |
| 102        | 200        |

---

## Gold Folder

```text
gold
│
├── customer_summary
├── sales_dashboard
├── finance_reports
└── inventory_kpis
```

---

# End-to-End Example

## Source

```text
SQL Server

↓

Orders
```

---

## Bronze

```text
Orders_Raw
```

Contains:

* Duplicate rows
* Invalid records
* Missing values

---

## Silver

Transformations:

* Remove duplicates
* Fix date formats
* Validate records
* Join customer information

Output:

```text
Orders_Clean
```

---

## Gold

Create:

```text
Daily Sales

Monthly Revenue

Customer KPIs

Top Products
```

Power BI connects to Gold.

---

# Complete Data Flow

```text
SQL Server
Oracle
SAP
REST API
CSV Files
      │
      ▼
Azure Data Factory
      │
      ▼
Azure Data Lake
      │
      ▼
Bronze
(Raw Data)
      │
      ▼
Databricks
      │
 Remove duplicates
 Handle NULLs
 Validate records
 Merge changes
 Standardize formats
      │
      ▼
Silver
(Clean Data)
      │
 Business Logic
 Aggregations
 KPI Calculations
 Star Schema
      │
      ▼
Gold
(Business Data)
      │
      ▼
Power BI
Machine Learning
SQL Analytics
```

---

# Real-Time Use Case 1 – Banking

## Bronze

Stores:

* ATM transactions
* Online banking
* UPI
* Card transactions

Exactly as received.

---

## Silver

* Remove duplicate transactions
* Validate account numbers
* Standardize timestamps
* Enrich with branch information

---

## Gold

Create:

* Daily deposits
* Loan summaries
* Customer balances
* Fraud dashboards

---

# Real-Time Use Case 2 – Retail

### Bronze

```text
Website Orders

POS Orders

Mobile Orders
```

---

### Silver

* Standardize product IDs
* Merge customer records
* Remove duplicate orders

---

### Gold

Business reports:

* Top-selling products
* Revenue by region
* Sales by month
* Customer lifetime value

---

# Real-Time Use Case 3 – Healthcare

### Bronze

* Patient registration
* Lab reports
* Medical history

---

### Silver

* Validate patient IDs
* Standardize diagnosis codes
* Remove duplicate patient records

---

### Gold

Hospital dashboards:

* Daily admissions
* ICU occupancy
* Disease trends
* Doctor performance

---

# Real-Time Use Case 4 – Manufacturing

### Bronze

Machine sensor logs.

---

### Silver

* Remove corrupt records
* Convert timestamps
* Filter invalid readings

---

### Gold

KPIs:

* Machine uptime
* Production efficiency
* Predictive maintenance indicators

---

# Why Delta Lake is Used in Medallion Architecture

Each layer is typically stored as **Delta Tables** because Delta Lake provides:

* ACID transactions
* Time Travel
* Schema Evolution
* MERGE support
* UPDATE and DELETE
* Fast reads
* Reliable writes

---

# Folder Structure in ADLS

```text
adls
│
├── bronze
│     ├── customer
│     ├── orders
│     └── products
│
├── silver
│     ├── customer
│     ├── orders
│     └── products
│
└── gold
      ├── sales_summary
      ├── customer_kpi
      └── finance_dashboard
```

---

# Bronze vs Silver vs Gold

| Feature         | Bronze  | Silver    | Gold                               |
| --------------- | ------- | --------- | ---------------------------------- |
| Data Quality    | Raw     | Cleaned   | Business-ready                     |
| Duplicates      | Present | Removed   | Removed                            |
| Transformations | Minimal | Extensive | Business logic                     |
| Aggregations    | No      | Limited   | Yes                                |
| Reporting       | No      | Sometimes | Yes                                |
| ML              | No      | Often     | Yes (features or curated datasets) |
| Power BI        | Rarely  | Sometimes | Usually                            |

---

# Real Enterprise Pipeline

```text
SAP
SQL Server
Oracle
CSV Files
REST APIs
      │
      ▼
Azure Data Factory
      │
      ▼
ADLS Bronze
      │
      ▼
Azure Databricks
      │
      ▼
Delta Silver
      │
      ▼
Delta Gold
      │
      ▼
Unity Catalog
      │
      ▼
Power BI
Azure ML
Databricks SQL
```

---

# Advantages of Medallion Architecture

* Improves data quality gradually.
* Keeps a raw copy for recovery and auditing.
* Separates ingestion from business logic.
* Makes pipelines easier to maintain.
* Supports incremental processing.
* Enables reliable analytics and machine learning.
* Reduces the impact of bad source data.

---

# Best Practices

1. **Bronze**

   * Never overwrite raw data unless required.
   * Preserve the original schema where possible.
   * Capture ingestion metadata (load time, source file, batch ID).

2. **Silver**

   * Apply validation rules.
   * Remove duplicates.
   * Handle late-arriving data.
   * Standardize formats and data types.

3. **Gold**

   * Design for business consumption.
   * Build fact and dimension tables if using a star schema.
   * Optimize for BI and reporting workloads.

---

# Interview Questions

### 1. What is Medallion Architecture?

**Answer:**

Medallion Architecture is a layered data architecture that organizes data into Bronze (raw), Silver (cleaned), and Gold (business-ready) layers to improve data quality, governance, and analytics.

---

### 2. Why do companies use Medallion Architecture?

* Preserves raw data for auditing and replay.
* Separates data cleansing from business logic.
* Produces trusted datasets for reporting.
* Simplifies maintenance and troubleshooting.

---

### 3. What happens in each layer?

| Layer  | Main Purpose                                         |
| ------ | ---------------------------------------------------- |
| Bronze | Ingest raw data from source systems                  |
| Silver | Clean, validate, enrich, and integrate data          |
| Gold   | Create business-ready datasets, KPIs, and aggregates |

---

### 4. Which layer is used by Power BI?

**Answer:**
Typically, **Gold** is used because it contains curated, business-ready datasets optimized for reporting. In some cases, advanced analysts or data scientists may also use **Silver** for detailed analysis.

---

# Easy Way to Remember

Think of a water purification process:

```text
River Water
    │
    ▼
Bronze
(Raw Water)
    │
    ▼
Silver
(Filtered Water)
    │
    ▼
Gold
(Purified Drinking Water)
```

* **Bronze** = Raw data from source systems.
* **Silver** = Cleaned and trusted data.
* **Gold** = Data ready for business decisions.

### One-line interview answer

> **Medallion Architecture is a layered data design pattern in Databricks that progressively improves data quality by storing raw data in Bronze, validated and enriched data in Silver, and business-ready, aggregated data in Gold for analytics, reporting, and machine learning.**
